# 16-01 从未来画面到机器人动作

本实验配合第 16 章的四个小章阅读：16-1 确认级联接口；本 Notebook 第 1 节对应 16-2 的未来计划，第 2 节对应 16-3 的动作转换，第 3、4 节对应 16-4 的换目标与执行反馈。我们固定同一个起点，先根据目标构造一段未来画面，再从画面里的夹爪位置得到数值动作，最后比较计划与实际观测。**画面由明确写出的规则构造，动作由颜色位置计算；这里没有训练视频生成器或 IDM，也不是 UniPi、AVDC 或 VERA 的复现。**

图中蓝块是夹爪的简化标记，橙块是杯子的简化标记，浅灰框是盒子。蓝块与橙块之间保留了绘图间距，以便两者都看得见；图像不能模拟真实抓取与碰撞。每个图像横向坐标是像素，不是机器人控制器中的米或弧度。

## 1. 当前画面和目标怎样形成未来计划

当前杯子在横向像素 22，目标盒子在像素 46。下面先规定四个时刻的画面：夹爪从 12 移到杯子附近的 22，闭合后与杯子一起移动，最后在盒子附近松开。这里使用人工规定的位置，是为了单独看清级联接口；真正的视频生成模块必须从数据中学习如何形成这种计划。运行后请先看蓝块和橙块的先后运动，不要把图像当作已经执行的录像。

In [ ]:
import numpy as np  # 保存图像和计划中的位置数值
import matplotlib.pyplot as plt  # 显示未来画面及后续对照结果
def draw_frame(gripper_x, cup_x, grip_closed, goal_x=46):  # 根据教学状态绘制一帧画面
    frame = np.full((64, 64, 3), 246, dtype=np.uint8)  # 创建浅色背景
    frame[28:40, goal_x-4:goal_x-3] = [177, 183, 188]  # 画出盒子的左边界
    frame[28:40, goal_x+3:goal_x+4] = [177, 183, 188]  # 画出盒子的右边界
    frame[28:29, goal_x-4:goal_x+4] = [177, 183, 188]  # 画出盒子的上边界
    frame[39:40, goal_x-4:goal_x+4] = [177, 183, 188]  # 画出盒子的下边界
    frame[31:36, cup_x-2:cup_x+3] = [207, 124, 63]  # 用橙色方块标记杯子
    frame[22:27, gripper_x-2:gripper_x+3] = [30, 95, 145]  # 用蓝色方块标记夹爪
    frame[27:28, gripper_x-1:gripper_x+2] = [30, 95, 145] if grip_closed else [246, 246, 246]  # 用短横线示意夹爪闭合
    return frame  # 返回可以直接显示的三通道图像
planned_gripper = np.array([12, 22, 34, 46])  # 指定四帧中的夹爪横向位置
planned_cup = np.array([22, 22, 34, 46])  # 指定四帧中的杯子横向位置
planned_closed = np.array([0, 1, 1, 0])  # 指定各帧夹爪的开合状态
planned_frames = [draw_frame(int(x), int(c), bool(k)) for x, c, k in zip(planned_gripper, planned_cup, planned_closed)]  # 按计划构造四帧
fig, axes = plt.subplots(1, 4, figsize=(12, 3))  # 创建四幅依次排列的画面
for step, (axis, frame) in enumerate(zip(axes, planned_frames)):  # 逐帧显示设想的未来
    axis.imshow(frame)  # 显示当前时刻的简化场景
    axis.set_title(f'Plan frame {step}')  # 标明计划中的时间顺序
    axis.axis('off')  # 隐藏不影响读图的坐标刻度
fig.tight_layout()  # 避免标题与画面重叠
plt.show()  # 输出第一阶段的视觉计划


### 怎样读这四帧

第 0 帧是当前起点，后面三帧是**设想**：先靠近并闭合夹爪，再把杯子移向目标，最后松开。浅灰框始终标记盒子的位置，它没有作为动作数值直接输入控制器。注意第 1 帧杯子尚未移动，而蓝块已经靠近；仅给出第 3 帧的终点，便看不出这一段必要的先后顺序。真实视频生成器还要处理接触、遮挡与物体形状，本实验的方块画面不能保证这些物理关系。

## 2. 从画面中读取夹爪轨迹，再得到动作

第二阶段不能把整张 RGB 图像直接发送给机械臂。这里先找到每帧的蓝色区域，取它的横向中心，再计算相邻两帧中心的差。所得的 `dx` 仍以**像素/计划步**计量；真实机器人还需要相机标定、坐标转换和动作边界检查。夹爪开合指令则取下一帧的计划状态。读图时对照第一张图的蓝块移动距离，检查三个 `dx` 是否合理。

In [ ]:
def blue_center_x(frame):  # 从一帧图像中找到蓝色夹爪的横向中心
    blue_mask = np.all(frame == np.array([30, 95, 145]), axis=2)  # 找到与蓝色标记完全相同的像素
    _, xs = np.nonzero(blue_mask)  # 读取蓝色像素的横向位置
    return float(xs.mean())  # 用像素均值代表夹爪中心
measured_gripper = np.array([blue_center_x(frame) for frame in planned_frames])  # 从画面而非原始位置数组读取轨迹
action_dx = np.diff(measured_gripper)  # 计算相邻计划画面之间的夹爪位移
action_grip = planned_closed[1:]  # 把下一帧开合状态作为该步命令
fig, axes = plt.subplots(1, 2, figsize=(10, 3))  # 分开显示计划位置与提取出的动作
axes[0].plot(range(4), measured_gripper, marker='o', color='#1e658f')  # 绘制每帧夹爪的横向位置
axes[0].set(xlabel='Plan frame', ylabel='Gripper x (pixels)', title='Visual plan')  # 标明位置的单位和含义
axes[1].bar(range(1, 4), action_dx, color='#9a7146')  # 绘制每一步从图像得到的位移命令
axes[1].set(xlabel='Action step', ylabel='dx (pixels/step)', title='Extracted motion')  # 标明位移尚未转换为物理单位
fig.tight_layout()  # 避免两个子图的标签重叠
plt.show()  # 输出位置计划与动作之间的对应关系
print('逐步动作（像素位移, 下一帧夹爪闭合=1）：', list(zip(action_dx.tolist(), action_grip.tolist())))  # 给出每一步的数值以便核对
assert np.allclose(action_dx, [10, 12, 12])  # 核对图像提取确实得到预期位移


### 两幅图和一行数值分别说明什么

左图的四个点来自四张计划画面，不是预先写好的动作标签；右图的三根柱子是相邻点之差，所以四帧只对应三步位移。第一步为 10 像素，第二、三步各为 12 像素。打印行还给出每一步结束时希望夹爪闭合还是张开：`1` 表示闭合，`0` 表示张开。这里的颜色追踪是特意简化的几何转换，说明第二阶段**可以不训练 IDM**；但像素位移不能直接用于真机，真实系统需要把图像坐标变为末端或关节命令，并验证该命令是否可执行。

## 3. 保持起点不变，换一个目标

级联式结构的第一阶段接收任务目标。现在把目标盒子改到像素 14，保持当前蓝块和橙块位置不变。人工规则会给出另一段未来，再用**同一个**图像位置提取方法转换成动作。这个对照要检查的是“目标先改变未来，未来再改变动作”，不是让模型对两段已经给定的候选动作打分。

In [ ]:
left_gripper = np.array([12, 22, 18, 14])  # 为新目标规定夹爪先靠近再向左运动
left_cup = np.array([22, 22, 18, 14])  # 为新目标规定杯子跟随向左移动
left_closed = np.array([0, 1, 1, 0])  # 保持先闭合再松开的动作阶段
left_frames = [draw_frame(int(x), int(c), bool(k), goal_x=14) for x, c, k in zip(left_gripper, left_cup, left_closed)]  # 构造左侧目标对应的未来画面
left_measured = np.array([blue_center_x(frame) for frame in left_frames])  # 读取新视频里的蓝块位置
left_dx = np.diff(left_measured)  # 从新视频得到新的位移命令
fig, axes = plt.subplots(2, 4, figsize=(12, 5))  # 上下对照右侧目标和左侧目标
for step in range(4):  # 依次显示两条计划的同一时刻
    axes[0, step].imshow(planned_frames[step])  # 上行显示原来的右侧目标计划
    axes[1, step].imshow(left_frames[step])  # 下行显示新的左侧目标计划
    axes[0, step].set_title(f'Right goal, frame {step}')  # 标记上行的目标与时间
    axes[1, step].set_title(f'Left goal, frame {step}')  # 标记下行的目标与时间
    axes[0, step].axis('off')  # 隐藏上行无关的图像刻度
    axes[1, step].axis('off')  # 隐藏下行无关的图像刻度
fig.tight_layout()  # 留出两行标题空间
plt.show()  # 显示目标变化首先如何改变未来画面
print('右侧目标的 dx：', action_dx.tolist())  # 显示原目标对应的位移
print('左侧目标的 dx：', left_dx.tolist())  # 显示新目标对应的位移
assert np.array_equal(planned_frames[0][22:27, 10:15], left_frames[0][22:27, 10:15])  # 核对两种任务的当前夹爪相同


### 应比较哪一段

两行的第 0 帧具有相同的杯子和夹爪起点，差别是目标盒子的位置。到第 2、3 帧，上行的杯子向右，下行的杯子向左。两条计划的第一步都先让夹爪靠近杯子，因此打印出的第一项相同；后两项一正一负，才分别把杯子送往不同目标。这个现象说明任务目标不是直接与一串动作数值比较，而是先影响未来计划，再通过动作接口影响最终命令。

## 4. 计划中的未来与真实观测可能不同

设想机器人只执行右侧目标计划的前两步。夹爪按命令到了像素 34，但杯子在搬运时滑动，实际只到像素 30；原计划第 2 帧却把杯子画在像素 34。下面把两张图并排放置。新一轮应读取右图的真实杯子位置，再形成新的未来，不能把左图的杯子位置当作真实起点。这里人为注入差异，只用于说明为什么要重新观察。

In [ ]:
imagined_frame = planned_frames[2]  # 取原计划中执行两步后设想的画面
observed_frame = draw_frame(34, 30, True)  # 人为构造杯子滑动后的实际观测
fig, axes = plt.subplots(1, 2, figsize=(7, 3))  # 创建计划与观测的并排视图
axes[0].imshow(imagined_frame)  # 左图显示计划以为会出现的画面
axes[1].imshow(observed_frame)  # 右图显示新的实际观测
axes[0].set_title('Imagined after 2 steps')  # 标记左图尚非事实
axes[1].set_title('Observed after 2 steps')  # 标记右图是新的起点
axes[0].axis('off')  # 隐藏左图无关的坐标刻度
axes[1].axis('off')  # 隐藏右图无关的坐标刻度
fig.tight_layout()  # 防止图像标题重叠
plt.show()  # 输出执行后的计划与观测对照
print('计划中的杯子 x =', planned_cup[2], '；重新观察到的杯子 x =', 30)  # 用数值说明重新起步的位置差异


### 这里的反馈属于哪一部分

左图与右图的夹爪位置都是 34，差异集中在橙色杯子：计划是 34，观测是 30。如果继续直接执行旧计划的剩余位移，系统会从错误的杯子起点做决定。正确的数据顺序是：读取右图作为当前观测，结合原目标再生成未来，再转换成下一段动作。这个实验没有实现真实机器人控制，也没有证明重新规划一定成功；它只核对级联式模型反复使用时哪一帧是事实、哪一帧是计划。